# Advanced Problems with Solutions: Closure Applications

Topics: closure state, `nonlocal`, free variables, function wrappers, counters, shared registries, metadata preservation, and decorator-ready designs.

## Problem 1 — Independent Stateful Counters

Write a function `make_counter(start=0, step=1)` that returns a closure. Each call should increment the internal value by `step` and return the new value. Each counter must maintain independent state.

In [1]:
def make_counter(start=0, step=1):
    value = start

    def counter():
        nonlocal value
        value += step
        return value

    return counter


c1 = make_counter(0, 1)
c2 = make_counter(100, 10)

assert c1() == 1
assert c1() == 2
assert c2() == 110
assert c2() == 120
assert c1() == 3

print('Problem 1 passed')

Problem 1 passed


### Solution Notes

`value` is a nonlocal variable captured by the inner function. Every call to `make_counter` creates a new local environment, so `c1` and `c2` do not share the same `value`.

## Problem 2 — Inspect Closure Variables

Create a counter closure and inspect its free variables using `.__code__.co_freevars` and `.__closure__`. Return a dictionary mapping free variable names to their current values.

In [2]:
def closure_vars(fn):
    names = fn.__code__.co_freevars
    cells = fn.__closure__ or ()
    return {name: cell.cell_contents for name, cell in zip(names, cells)}


c = make_counter(10, 5)
assert closure_vars(c) == {'step': 5, 'value': 10}

c()
assert closure_vars(c) == {'step': 5, 'value': 15}

print(closure_vars(c))

{'step': 5, 'value': 15}


### Solution Notes

`co_freevars` contains the names of captured variables. `__closure__` contains cell objects holding the current values of those variables.

## Problem 3 — Counting Calls Without Globals

Write `count_calls(fn, registry)` that returns a wrapper. The wrapper should:

- call the original function,
- count how many times that wrapped function was called,
- store the count in the provided `registry` dictionary,
- support arbitrary positional and keyword arguments.

In [3]:
def count_calls(fn, registry):
    count = 0

    def inner(*args, **kwargs):
        nonlocal count
        count += 1
        registry[fn.__name__] = count
        return fn(*args, **kwargs)

    return inner


def add(a, b):
    return a + b


def power(base, exponent=2):
    return base ** exponent


calls = {}
add_counted = count_calls(add, calls)
power_counted = count_calls(power, calls)

assert add_counted(2, 3) == 5
assert add_counted(10, 20) == 30
assert power_counted(3) == 9
assert power_counted(2, exponent=5) == 32
assert calls == {'add': 2, 'power': 2}

print(calls)

{'add': 2, 'power': 2}


### Solution Notes

The registry is captured as a nonlocal variable, not accessed as a global. This makes the wrapper reusable and easier to test.

## Problem 4 — Metadata Preservation

Improve the previous wrapper so that the wrapped function preserves the original function's name and docstring. Use `functools.wraps`.

In [4]:
from functools import wraps


def count_calls_preserve_metadata(fn, registry):
    count = 0

    @wraps(fn)
    def inner(*args, **kwargs):
        nonlocal count
        count += 1
        registry[fn.__name__] = count
        return fn(*args, **kwargs)

    return inner


def factorial(n):
    """Return n factorial for n >= 0."""
    result = 1
    for i in range(2, n + 1):
        result *= i
    return result


registry = {}
factorial = count_calls_preserve_metadata(factorial, registry)

assert factorial.__name__ == 'factorial'
assert factorial.__doc__ == 'Return n factorial for n >= 0.'
assert factorial(5) == 120
assert registry == {'factorial': 1}

print(factorial.__name__, factorial.__doc__, registry)

factorial Return n factorial for n >= 0. {'factorial': 1}


### Solution Notes

Without `@wraps(fn)`, the returned function would usually be named `inner`, which can make debugging, introspection, and documentation worse.

## Problem 5 — Build a Decorator Factory

Write a decorator factory `call_counter(registry)` so functions can be decorated using:

```python
@call_counter(registry)
def my_function(...):
    ...
```

In [5]:
def call_counter(registry):
    def decorator(fn):
        count = 0

        @wraps(fn)
        def inner(*args, **kwargs):
            nonlocal count
            count += 1
            registry[fn.__name__] = count
            return fn(*args, **kwargs)

        return inner

    return decorator


stats = {}


@call_counter(stats)
def multiply(a, b, c=1):
    return a * b * c


@call_counter(stats)
def greet(name, greeting='Hello'):
    return f'{greeting}, {name}!'


assert multiply(2, 3) == 6
assert multiply(2, 3, c=4) == 24
assert greet('Ada') == 'Hello, Ada!'
assert greet('Grace', greeting='Hi') == 'Hi, Grace!'
assert stats == {'multiply': 2, 'greet': 2}

print(stats)

{'multiply': 2, 'greet': 2}


### Solution Notes

`call_counter` closes over `registry`. `decorator` closes over `registry`. `inner` closes over `count`, `fn`, and `registry`.

## Problem 6 — Per-Function and Total Counts

Extend the decorator factory so the registry tracks both individual function counts and a total call count across all decorated functions.

Expected registry shape:

```python
{
    'total': 3,
    'functions': {
        'f': 2,
        'g': 1
    }
}
```

In [6]:
def advanced_call_counter(registry):
    registry.setdefault('total', 0)
    registry.setdefault('functions', {})

    def decorator(fn):
        count = 0

        @wraps(fn)
        def inner(*args, **kwargs):
            nonlocal count
            count += 1
            registry['total'] += 1
            registry['functions'][fn.__name__] = count
            return fn(*args, **kwargs)

        return inner

    return decorator


metrics = {}


@advanced_call_counter(metrics)
def square(x):
    return x * x


@advanced_call_counter(metrics)
def cube(x):
    return x * x * x


assert square(2) == 4
assert square(3) == 9
assert cube(2) == 8

assert metrics == {
    'total': 3,
    'functions': {
        'square': 2,
        'cube': 1
    }
}

print(metrics)

{'total': 3, 'functions': {'square': 2, 'cube': 1}}


### Solution Notes

The per-function `count` lives in each wrapper closure. The shared total lives in the shared registry.

## Problem 7 — Resettable Counter Closure

Create `make_resettable_counter(start=0)` that returns two closures: `inc` and `reset`.

- `inc(step=1)` increments and returns the current value.
- `reset(value=start)` resets the internal value and returns it.
- Both closures must share the same internal state.

In [7]:
def make_resettable_counter(start=0):
    current = start

    def inc(step=1):
        nonlocal current
        current += step
        return current

    def reset(value=start):
        nonlocal current
        current = value
        return current

    return inc, reset


inc, reset = make_resettable_counter(10)

assert inc() == 11
assert inc(9) == 20
assert reset() == 10
assert inc() == 11
assert reset(100) == 100
assert inc(50) == 150

print('Problem 7 passed')

Problem 7 passed


### Solution Notes

`inc` and `reset` are separate closures, but they close over the same `current` cell.

## Problem 8 — Call History Logger

Write a decorator factory `record_calls(history)` that records every call as a dictionary containing:

- function name,
- positional arguments,
- keyword arguments,
- returned result.

The wrapper should still return the original result.

In [8]:
def record_calls(history):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            result = fn(*args, **kwargs)
            history.append({
                'function': fn.__name__,
                'args': args,
                'kwargs': kwargs,
                'result': result
            })
            return result

        return inner

    return decorator


history = []


@record_calls(history)
def divide(a, b):
    return a / b


assert divide(10, 2) == 5
assert divide(9, b=3) == 3

assert history == [
    {'function': 'divide', 'args': (10, 2), 'kwargs': {}, 'result': 5.0},
    {'function': 'divide', 'args': (9,), 'kwargs': {'b': 3}, 'result': 3.0}
]

print(history)

[{'function': 'divide', 'args': (10, 2), 'kwargs': {}, 'result': 5.0}, {'function': 'divide', 'args': (9,), 'kwargs': {'b': 3}, 'result': 3.0}]


### Solution Notes

This is a practical closure use case: adding behavior around a function without changing the function's own implementation.

## Problem 9 — Handle Exceptions in a Wrapper

Modify the call-history idea so that exceptions are logged too. If the wrapped function raises an exception, the wrapper should log the exception type and then re-raise the exception.

In [9]:
def record_calls_and_errors(history):
    def decorator(fn):
        @wraps(fn)
        def inner(*args, **kwargs):
            try:
                result = fn(*args, **kwargs)
            except Exception as ex:
                history.append({
                    'function': fn.__name__,
                    'args': args,
                    'kwargs': kwargs,
                    'exception': type(ex).__name__
                })
                raise
            else:
                history.append({
                    'function': fn.__name__,
                    'args': args,
                    'kwargs': kwargs,
                    'result': result
                })
                return result

        return inner

    return decorator


events = []


@record_calls_and_errors(events)
def safe_ratio(a, b):
    return a / b


assert safe_ratio(10, 5) == 2

try:
    safe_ratio(10, 0)
except ZeroDivisionError:
    pass

assert events[0]['result'] == 2
assert events[1]['exception'] == 'ZeroDivisionError'

print(events)

[{'function': 'safe_ratio', 'args': (10, 5), 'kwargs': {}, 'result': 2.0}, {'function': 'safe_ratio', 'args': (10, 0), 'kwargs': {}, 'exception': 'ZeroDivisionError'}]


### Solution Notes

A wrapper should not accidentally swallow errors unless that is explicitly required. Logging and re-raising keeps the original behavior intact.

## Problem 10 — Closure-Based Memoization

Write a closure-based decorator `memoize(fn)` that caches results by argument tuple. Use it on a recursive Fibonacci function and track how many times the actual function body runs.

In [10]:
def memoize(fn):
    cache = {}

    @wraps(fn)
    def inner(*args):
        if args not in cache:
            cache[args] = fn(*args)
        return cache[args]

    inner.cache = cache
    return inner


body_runs = {'fib': 0}


@memoize
def fib(n):
    body_runs['fib'] += 1
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)


assert fib(10) == 55
assert body_runs['fib'] == 11
assert sorted(fib.cache.keys()) == [(0,), (1,), (2,), (3,), (4,), (5,), (6,), (7,), (8,), (9,), (10,)]

print('fib(10) =', fib(10))
print('actual body runs =', body_runs['fib'])
print('cache size =', len(fib.cache))

fib(10) = 55
actual body runs = 11
cache size = 11


### Solution Notes

`cache` is closure state. The function remembers previous results without using a global variable or a class.